# ML-04 — Search Intelligence Data Contract

**Lane:** Content Refresh & Performance Prediction  
**Goal:** Identify content items showing signs of declining search performance early enough to refresh them.


## 1) Contract

### 1.1 What does one row mean?
One row represents one **client-content item pair** for the selected feature/decision window. The content item is identified by `content_hash_id` and scoped to a client using `client_hash_id`.

### 1.2 Which tables will I use?
- `fact_content_daily_performance` for daily Google Search Console performance.
- `fact_content_query_90d` for query-mix and concentration signals.
- `dim_clients` for client-level history/context when needed.

### 1.3 What time window will I use?
I use a **mid-panel month (March 2026)** for development and verification. I avoid the final-month `_sample` table because June 2026 is the natural outcome window and could encourage label leakage.

### 1.4 What will I predict/rank?
I will predict whether a content item experiences a **more than 20% decline in impressions in the following outcome window**, using only information available before that outcome window.

### 1.5 What will I deliberately exclude?
I deliberately exclude future-window impressions, clicks, CTR, position, and any other outcome-derived value from the predictive features. These would leak the answer into the model.


## 2) Connect to the full Hugging Face release

This notebook reads Parquet through DuckDB and only materializes small query results into pandas. Store your Hugging Face token as a Colab Secret named `HF_TOKEN` (or use the prompt as a last resort).

In [ ]:
%pip -q install duckdb huggingface_hub pandas scikit-learn

import os, getpass

HF_TOKEN = os.environ.get('HF_TOKEN')
if not HF_TOKEN:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get('HF_TOKEN')
    except Exception:
        pass
HF_TOKEN = HF_TOKEN or getpass.getpass('Paste your Hugging Face READ token (hf_...): ')
assert HF_TOKEN, 'HF_TOKEN is required.'


In [ ]:
import duckdb
   
con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")
   
REL = 'hf://datasets/FlyRank/internship-warehouse'
   
TABLES = {
    'dim_clients': f"read_parquet('{REL}/dim_clients.parquet')",
    'dim_content': f"read_parquet('{REL}/dim_content.parquet')",
    'fact_daily': f"read_parquet('{REL}/fact_content_daily_performance/**/*.parquet')",
    'fact_daily_sample': f"read_parquet('{REL}/fact_content_daily_performance_sample.parquet')",
    'fact_query_90d': f"read_parquet('{REL}/fact_content_query_90d.parquet')",
}

print('Connected to FlyRank internship warehouse.')
for name, src in TABLES.items():
    print(name, '->', src)


## 3) Exactly three verification queries

The following three queries verify the grain, the March 2026 slice, and availability. The availability check intentionally uses `IS TRUE` as required by the assignment.

### Query 1 — Grain
The intended daily fact grain is one row per `client_hash_id + content_hash_id + report_date`. The query checks whether that combination is unique.

In [ ]:
q1 = con.sql(f"""
SELECT
    COUNT(*) AS total_rows,
    COUNT(DISTINCT client_hash_id || '|' || content_hash_id || '|' || CAST(report_date AS VARCHAR)) AS distinct_grain_keys,
    COUNT(*) - COUNT(DISTINCT client_hash_id || '|' || content_hash_id || '|' || CAST(report_date AS VARCHAR)) AS duplicate_rows
FROM {TABLES['fact_daily']}
WHERE report_date >= DATE '2026-03-01'
  AND report_date <  DATE '2026-04-01'
""").df()
display(q1)


### Query 2 — March 2026 row count and date span

In [ ]:
q2 = con.sql(f"""
SELECT
    COUNT(*) AS row_count,
    MIN(report_date) AS min_date,
    MAX(report_date) AS max_date
FROM {TABLES['fact_daily']}
WHERE report_date >= DATE '2026-03-01'
  AND report_date <  DATE '2026-04-01'
""").df()
display(q2)


### Query 3 — Availability using `IS TRUE`
This checks the March 2026 slice using an availability/validity field when present. The notebook discovers the appropriate boolean availability column before running the required `IS TRUE` filter.

In [ ]:
schema = con.sql(f"DESCRIBE SELECT * FROM {TABLES['fact_daily']}").df()
bool_cols = [r['column_name'] for _, r in schema.iterrows() if str(r['column_type']).upper() in ('BOOLEAN', 'BOOL')]
preferred = [c for c in bool_cols if any(k in c.lower() for k in ['available','valid','observed','gsc'])]
availability_col = preferred[0] if preferred else (bool_cols[0] if bool_cols else None)

print('Boolean columns:', bool_cols)
print('Selected availability column:', availability_col)

if availability_col:
    q3 = con.sql(f"""
    SELECT
        COUNT(*) AS march_rows,
        COUNT(*) FILTER (WHERE {availability_col} IS TRUE) AS available_rows
    FROM {TABLES['fact_daily']}
    WHERE report_date >= DATE '2026-03-01'
      AND report_date <  DATE '2026-04-01'
    """).df()
else:
    q3 = con.sql(f"""
    SELECT
        COUNT(*) AS march_rows,
        NULL::BIGINT AS available_rows
    FROM {TABLES['fact_daily']}
    WHERE report_date >= DATE '2026-03-01'
      AND report_date <  DATE '2026-04-01'
    """).df()
    print('No BOOLEAN availability column was exposed in the schema; inspect the schema before final submission.')

display(q3)


## 4) Five-feature frame

The feature frame uses a March 2026 decision point. Features are calculated from the **30 days before the decision point** so they are available before the outcome window.

| Feature | Available when? |
|---|---|
| `imp_prev30` | Known after the previous 30 days have been observed, before the outcome window. |
| `clk_prev30` | Known after the previous 30 days have been observed, before the outcome window. |
| `ctr_prev30` | Calculated from previous-period clicks and impressions only. |
| `pos_prev30` | Average search position from the previous 30 days only. |
| `query_concentration` | Derived from historical 90-day query impressions available at the decision point. |


In [ ]:
features = con.sql(f"""
WITH base AS (
    SELECT
        client_hash_id,
        content_hash_id,
        SUM(gsc_impressions) AS imp_prev30,
        SUM(gsc_clicks) AS clk_prev30,
        AVG(gsc_avg_position) AS pos_prev30
    FROM {TABLES['fact_daily']}
    WHERE report_date >= DATE '2026-02-01'
      AND report_date <  DATE '2026-03-01'
    GROUP BY 1, 2
    HAVING SUM(gsc_impressions) >= 100
),
queries AS (
    SELECT
        content_hash_id,
        SUM(impressions_90d) AS kept_impressions,
        MAX(impressions_90d) AS top_query_impressions
    FROM {TABLES['fact_query_90d']}
    GROUP BY content_hash_id
)
SELECT
    b.client_hash_id,
    b.content_hash_id,
    b.imp_prev30,
    b.clk_prev30,
    CASE WHEN b.imp_prev30 > 0 THEN b.clk_prev30 / b.imp_prev30 ELSE NULL END AS ctr_prev30,
    b.pos_prev30,
    CASE WHEN q.kept_impressions > 0 THEN q.top_query_impressions / q.kept_impressions ELSE NULL END AS query_concentration
FROM base b
LEFT JOIN queries q USING (content_hash_id)
""").df()

print(f'Feature rows: {len(features):,}')
display(features.head())


## 5) Deliberate leakage experiment

The honest label is a future outcome: whether March 2026 impressions decline by more than 20% compared with February 2026. The deliberately leaked feature is computed directly from that future outcome. It is included only to demonstrate why a suspiciously high score is not trustworthy.

In [ ]:
outcome = con.sql(f"""
SELECT
    client_hash_id,
    content_hash_id,
    SUM(gsc_impressions) AS imp_future30
FROM {TABLES['fact_daily']}
WHERE report_date >= DATE '2026-03-01'
  AND report_date <  DATE '2026-04-01'
GROUP BY 1, 2
""").df()

data = features.merge(outcome, on=['client_hash_id','content_hash_id'], how='inner')
data['is_declining'] = (data['imp_future30'] < 0.8 * data['imp_prev30']).astype(int)

print('Rows with both feature and outcome windows:', len(data))
print('Decline rate:', round(data['is_declining'].mean(), 3))


In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report

honest_cols = ['imp_prev30','clk_prev30','ctr_prev30','pos_prev30','query_concentration']
model_data = data.dropna(subset=honest_cols + ['is_declining']).copy()

X = model_data[honest_cols]
y = model_data['is_declining']

X_tr, X_te, y_tr, y_te = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y
)

honest_model = RandomForestClassifier(
    n_estimators=150, random_state=42, n_jobs=-1
).fit(X_tr, y_tr)

print('HONEST MODEL')
print('Base rate:', round(max(y_te.mean(), 1-y_te.mean()), 3))
print(classification_report(y_te, honest_model.predict(X_te), digits=3))


In [ ]:
# Deliberate leakage: this feature uses the future outcome itself.
model_data['leak_future_decline'] = model_data['is_declining']

leak_cols = honest_cols + ['leak_future_decline']
X_leak = model_data[leak_cols]
   
Xl_tr, Xl_te, yl_tr, yl_te = train_test_split(
    X_leak, y, test_size=0.25, random_state=42, stratify=y
)

leak_model = RandomForestClassifier(
    n_estimators=150, random_state=42, n_jobs=-1
).fit(Xl_tr, yl_tr)

print('LEAKED MODEL — intentionally invalid')
print(classification_report(yl_te, leak_model.predict(Xl_te), digits=3))


### Leakage lesson

The leaked model can become unrealistically accurate because `leak_future_decline` directly contains information about the label. This feature is **not available at the decision moment**, so it must never be used for production prediction. I remove it and keep the honest feature set only.

**Removed:** `leak_future_decline`.

In [ ]:
data = data.drop(columns=['is_declining'], errors='ignore')
print('Final retained features:', honest_cols)
print('Leakage feature retained? ', 'leak_future_decline' in data.columns)


## 6) Limitation

**Limitation:** the warehouse is an unbalanced panel, so clients have different amounts of historical data. A feature window that is available for one client may not be available for another. The model should therefore enforce minimum-history requirements and be evaluated with client-level splits before being treated as generalizable.


## 7) Self-check

- [x] Five plain-words contract answers provided.
- [x] Exactly three verification query sections provided.
- [x] Availability query uses `IS TRUE` when a boolean availability field is exposed.
- [x] Five features with an availability explanation provided.
- [x] Deliberate leakage experiment shown.
- [x] Leakage feature removed from the retained feature set.
- [x] One named limitation provided.

**Before committing:** execute every code cell in Colab and verify that the three query outputs and model outputs are visible in the saved notebook.